In [20]:
import os
import numpy as np
import re
import random
import time
from copy import deepcopy
import pandas as pd
from datetime import datetime

In [21]:
class VSBPP:
    def __init__(self):
        self.num_items = 0
        self.num_bin_types = 0
        self.instance_id = 0
        self.bin_capacities = []
        self.bin_costs = []
        self.item_weights = []
        
    def read_instance(self, file_path):
        """
        Lê uma instância do problema VSBPP a partir de um arquivo.
        """
        try:
            with open(file_path, 'r') as f:
                # Lê a primeira linha com informações básicas
                first_line = f.readline().strip().split()
                self.num_items = int(first_line[0])
                self.num_bin_types = int(first_line[1])
                self.instance_id = int(first_line[3]) if len(first_line) > 3 else 0
                
                # Lê a segunda linha com capacidades e custos dos bins
                second_line = f.readline().strip().split()
                
                # Se tivermos mais números do que 2*num_bin_types, significa que 
                # temos capacidades e custos intercalados
                if len(second_line) >= 2 * self.num_bin_types:
                    for i in range(self.num_bin_types):
                        self.bin_capacities.append(int(second_line[i*2]))
                        self.bin_costs.append(int(second_line[i*2+1]))
                else:
                    # Caso contrário, assumimos que estão em blocos separados
                    self.bin_capacities = [int(x) for x in second_line[:self.num_bin_types]]
                    self.bin_costs = [int(x) for x in second_line[self.num_bin_types:]]
                
                # Lê os pesos dos itens (podem estar em várias linhas)
                remaining_content = f.read()
                # Remove espaços extras e quebras de linha
                cleaned_content = re.sub(r'\s+', ' ', remaining_content).strip()
                item_weights_str = cleaned_content.split()
                
                self.item_weights = [int(w) for w in item_weights_str]
                
                # Verifica se o número de itens está correto
                if len(self.item_weights) != self.num_items:
                    print(f"Aviso: Número esperado de itens ({self.num_items}) é diferente do número lido ({len(self.item_weights)})")
                    
                return True
        except Exception as e:
            print(f"Erro ao ler a instância: {e}")
            return False
    
    def print_instance(self):
        """
        Imprime informações detalhadas sobre a instância no formato solicitado.
        """
        print(f"--------------------- INSTANCIA: {self.instance_id} ---------------------")
        print(f"Numero de itens (n): {self.num_items}")
        print(f"Numero de tipos de bin (m): {self.num_bin_types}")
        print()
        print("Tipos de bin (capacidade / custo):")
        for i in range(self.num_bin_types):
            print(f"  Tipo {i+1}: {self.bin_capacities[i]} / {self.bin_costs[i]}")
        print()
        print("Pesos dos itens:")
        items_per_line = 20
        for i in range(0, self.num_items, items_per_line):
            print(" ".join(str(w) for w in self.item_weights[i:i+items_per_line]))
        print()
        print("--------------------------------")

In [22]:
def load_all_instances(directory_path):
    """
    Carrega todas as instâncias de uma pasta.
    """
    instances = []
    
    for filename in os.listdir(directory_path):
        if filename.endswith('.txt'):
            file_path = os.path.join(directory_path, filename)
            instance = VSBPP()
            if instance.read_instance(file_path):
                instances.append((filename, instance))
                print(f"Instância carregada: {filename}")
    
    return instances

In [23]:
# Função para calcular o custo total de uma solução
def calculate_total_cost(instance, bin_types):
    """
    Calcula o custo total de uma solução.
    """
    return sum(instance.bin_costs[bin_type] for bin_type in bin_types)

In [24]:
# Função para encontrar o menor tipo de bin que pode acomodar um peso
def find_smallest_bin_type(instance, weight):
    """
    Encontra o menor tipo de bin que pode acomodar um peso.
    """
    for i in range(instance.num_bin_types):
        if instance.bin_capacities[i] >= weight:
            return i
    return -1  # Não deveria acontecer se a instância for válida

In [25]:
# Implementação da heurística Best Fit Decreasing (BFD)
def best_fit_decreasing(instance):
    """
    Implementa a heurística Best Fit Decreasing para o VSBPP.
    """
    start_time = time.time()
    
    # Ordena os itens em ordem decrescente de peso
    sorted_items = sorted(enumerate(instance.item_weights), key=lambda x: x[1], reverse=True)
    
    # Inicializa a solução vazia
    bins = []  # Lista de bins usados
    bin_types = []  # Tipo de cada bin usado
    remaining_capacities = []  # Capacidade restante em cada bin
    
    # Para cada item (ordenado)
    for item_idx, item_weight in sorted_items:
        # Encontra o bin com melhor ajuste para o item atual
        best_bin_idx = -1
        best_remaining = float('inf')
        
        for i, capacity in enumerate(remaining_capacities):
            if capacity >= item_weight and capacity - item_weight < best_remaining:
                best_bin_idx = i
                best_remaining = capacity - item_weight
        
        # Se encontrou um bin adequado, insere o item nele
        if best_bin_idx != -1:
            bins[best_bin_idx].append(item_idx)
            remaining_capacities[best_bin_idx] -= item_weight
        else:
            # Se não encontrou um bin adequado, cria um novo bin
            # Escolhe o menor bin possível que acomoda o item
            bin_type = find_smallest_bin_type(instance, item_weight)
            
            # Se nenhum bin adequado foi encontrado (o que não deveria acontecer em instâncias válidas)
            if bin_type == -1:
                raise ValueError(f"Item {item_idx} com peso {item_weight} não cabe em nenhum tipo de bin disponível")
            
            # Cria um novo bin do tipo escolhido
            bins.append([item_idx])
            bin_types.append(bin_type)
            remaining_capacities.append(instance.bin_capacities[bin_type] - item_weight)
    
    # Calcula o custo total
    total_cost = calculate_total_cost(instance, bin_types)
    
    end_time = time.time()
    execution_time = end_time - start_time
    
    return bins, bin_types, total_cost, execution_time

In [26]:
# Implementação da heurística Subset-Sum Problem (SSP)
def subset_sum_heuristic(instance):
    """
    Implementa a heurística baseada no Problema da Soma de Subconjuntos (Subset Sum Problem).
    Esta heurística é considerada uma das melhores para o VSBPP de acordo com a literatura.
    """
    start_time = time.time()
    
    # Cria uma cópia dos itens para manipular
    remaining_items = [(i, weight) for i, weight in enumerate(instance.item_weights)]
    # Ordena os itens em ordem decrescente de peso
    remaining_items.sort(key=lambda x: x[1], reverse=True)
    
    bins = []  # Lista de bins usados
    bin_types = []  # Tipo de cada bin usado
    
    while remaining_items:
        # Para cada tipo de bin, encontramos o melhor subconjunto de itens
        best_bin_type = -1
        best_items = []
        best_ratio = float('inf')  # Razão custo/utilização
        
        for bin_type in range(instance.num_bin_types):
            capacity = instance.bin_capacities[bin_type]
            cost = instance.bin_costs[bin_type]
            
            # Usamos programação dinâmica para resolver o problema da mochila
            # para encontrar o melhor conjunto de itens para este bin
            selected_items = knapsack(remaining_items, capacity)
            
            # Calcula o peso total dos itens selecionados
            total_weight = sum(item[1] for item in selected_items)
            
            # Só consideramos se pelo menos um item foi selecionado
            if selected_items:
                # Calcula a razão custo/utilização
                ratio = cost / total_weight
                
                # Verifica se este é o melhor bin tipo até agora
                if ratio < best_ratio:
                    best_bin_type = bin_type
                    best_items = selected_items
                    best_ratio = ratio
        
        # Se não encontramos nenhum bin adequado, paramos
        if best_bin_type == -1:
            break
        
        # Cria um novo bin do tipo escolhido com os itens selecionados
        bin_items = [item[0] for item in best_items]
        bins.append(bin_items)
        bin_types.append(best_bin_type)
        
        # Remove os itens selecionados da lista de itens restantes
        for item in best_items:
            remaining_items.remove(item)
    
    # Calcula o custo total
    total_cost = calculate_total_cost(instance, bin_types)
    
    end_time = time.time()
    execution_time = end_time - start_time
    
    return bins, bin_types, total_cost, execution_time


In [27]:
# Função para resolver o problema da mochila usando programação dinâmica
def knapsack(items, capacity):
    """
    Resolve o problema da mochila para encontrar o melhor conjunto de itens
    que maximiza o peso total sem exceder a capacidade.
    
    Args:
        items: Lista de tuplas (índice, peso)
        capacity: Capacidade máxima
    
    Returns:
        Lista de itens selecionados
    """
    n = len(items)
    # Se não há itens ou a capacidade é 0, retorna lista vazia
    if n == 0 or capacity == 0:
        return []
    
    # Inicializa a tabela de programação dinâmica
    dp = [[0 for _ in range(capacity + 1)] for _ in range(n + 1)]
    
    # Preenche a tabela
    for i in range(1, n + 1):
        for w in range(1, capacity + 1):
            # Se o item atual pode ser incluído
            if items[i-1][1] <= w:
                dp[i][w] = max(items[i-1][1] + dp[i-1][w-items[i-1][1]], dp[i-1][w])
            else:
                dp[i][w] = dp[i-1][w]
    
    # Reconstrói a solução
    w = capacity
    selected = []
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i-1][w]:
            selected.append(items[i-1])
            w -= items[i-1][1]
    
    return selected

In [28]:
# Heurística de busca local 1: Troca de itens entre bins
def swap_items(instance, bins, bin_types):
    """
    Busca local que tenta trocar itens entre bins para reduzir o custo total.
    """
    start_time = time.time()
    
    improved = True
    total_cost = calculate_total_cost(instance, bin_types)
    
    while improved:
        improved = False
        
        # Para cada par de bins
        for i in range(len(bins)):
            for j in range(i+1, len(bins)):
                bin_i_type = bin_types[i]
                bin_j_type = bin_types[j]
                
                # Calcula a soma atual dos pesos em cada bin
                sum_i = sum(instance.item_weights[item] for item in bins[i])
                sum_j = sum(instance.item_weights[item] for item in bins[j])
                
                # Para cada par de itens nos dois bins
                for item_i_idx, item_i in enumerate(bins[i]):
                    for item_j_idx, item_j in enumerate(bins[j]):
                        # Peso dos itens
                        weight_i = instance.item_weights[item_i]
                        weight_j = instance.item_weights[item_j]
                        
                        # Verificamos se após a troca os bins ainda respeitam as capacidades
                        new_sum_i = sum_i - weight_i + weight_j
                        new_sum_j = sum_j - weight_j + weight_i
                        
                        if (new_sum_i <= instance.bin_capacities[bin_i_type] and 
                            new_sum_j <= instance.bin_capacities[bin_j_type]):
                            
                            # Fazemos a troca temporariamente
                            bins[i][item_i_idx], bins[j][item_j_idx] = bins[j][item_j_idx], bins[i][item_i_idx]
                            
                            # Verificamos se podemos usar tipos de bins menores após a troca
                            new_bin_i_type = find_smallest_bin_type(instance, new_sum_i)
                            new_bin_j_type = find_smallest_bin_type(instance, new_sum_j)
                            
                            # Calculamos o novo custo
                            new_cost = total_cost - instance.bin_costs[bin_i_type] - instance.bin_costs[bin_j_type] + instance.bin_costs[new_bin_i_type] + instance.bin_costs[new_bin_j_type]
                            
                            # Se o custo melhorou, aceitamos a solução
                            if new_cost < total_cost:
                                bin_types[i] = new_bin_i_type
                                bin_types[j] = new_bin_j_type
                                total_cost = new_cost
                                improved = True
                            else:
                                # Desfazemos a troca
                                bins[i][item_i_idx], bins[j][item_j_idx] = bins[j][item_j_idx], bins[i][item_i_idx]
                        
                        if improved:
                            break
                    if improved:
                        break
                if improved:
                    break
            if improved:
                break
    
    end_time = time.time()
    execution_time = end_time - start_time
    
    return bins, bin_types, total_cost, execution_time


In [29]:
# Heurística de busca local 2: Mudança de tipo de bin
def change_bin_type(instance, bins, bin_types):
    """
    Busca local que tenta mudar o tipo de cada bin para reduzir o custo total.
    """
    start_time = time.time()
    
    total_cost = calculate_total_cost(instance, bin_types)
    improved = True
    
    while improved:
        improved = False
        
        # Para cada bin
        for i in range(len(bins)):
            current_type = bin_types[i]
            
            # Calcula a soma dos pesos no bin
            bin_sum = sum(instance.item_weights[item] for item in bins[i])
            
            # Tenta todos os tipos de bin que podem acomodar esse peso
            for bin_type in range(instance.num_bin_types):
                if (bin_type != current_type and 
                    instance.bin_capacities[bin_type] >= bin_sum and 
                    instance.bin_costs[bin_type] < instance.bin_costs[current_type]):
                    
                    # Muda o tipo do bin
                    bin_types[i] = bin_type
                    total_cost = total_cost - instance.bin_costs[current_type] + instance.bin_costs[bin_type]
                    improved = True
                    break
    
    end_time = time.time()
    execution_time = end_time - start_time
    
    return bins, bin_types, total_cost, execution_time

In [30]:
# Heurística de busca local 3: Redistribuição de itens
def redistribute_items(instance, bins, bin_types):
    """
    Busca local que tenta redistribuir itens entre bins para reduzir o número de bins usados.
    """
    start_time = time.time()
    
    improved = True
    total_cost = calculate_total_cost(instance, bin_types)
    
    while improved:
        improved = False
        
        # Tenta cada bin como candidato para remoção
        for i in range(len(bins)):
            # Tenta redistribuir os itens do bin i para outros bins
            candidate_items = bins[i].copy()
            
            # Verifica se cada item pode ser inserido em outro bin
            all_items_placed = True
            new_bins = deepcopy(bins)
            new_bin_types = bin_types.copy()
            
            # Remove o bin candidato temporariamente
            del new_bins[i]
            del new_bin_types[i]
            
            for item in candidate_items:
                item_weight = instance.item_weights[item]
                item_placed = False
                
                # Tenta inserir o item em cada bin existente
                for j in range(len(new_bins)):
                    # Calcula a soma atual dos pesos no bin
                    bin_sum = sum(instance.item_weights[x] for x in new_bins[j])
                    
                    # Verifica se o item cabe no bin atual
                    if bin_sum + item_weight <= instance.bin_capacities[new_bin_types[j]]:
                        new_bins[j].append(item)
                        item_placed = True
                        break
                
                if not item_placed:
                    # Se o item não pôde ser colocado em nenhum bin existente,
                    # não podemos remover o bin candidato
                    all_items_placed = False
                    break
            
            # Se todos os itens foram redistribuídos, aceita a nova solução
            if all_items_placed:
                bins = new_bins
                bin_types = new_bin_types
                new_cost = calculate_total_cost(instance, bin_types)
                improved = True
                total_cost = new_cost
                break
    
    end_time = time.time()
    execution_time = end_time - start_time
    
    return bins, bin_types, total_cost, execution_time

In [31]:
# Implementação da meta-heurística Variable Neighborhood Descent (VND)
def variable_neighborhood_descent(instance, initial_bins, initial_bin_types):
    """
    Implementa a meta-heurística Variable Neighborhood Descent para o VSBPP.
    """
    start_time = time.time()
    
    # Define as estruturas de vizinhança
    neighborhoods = [
        swap_items,        # Troca de itens entre bins
        change_bin_type,   # Mudança de tipo de bin
        redistribute_items  # Redistribuição de itens
    ]
    
    # Inicializa a solução atual
    current_bins = deepcopy(initial_bins)
    current_bin_types = initial_bin_types.copy()
    current_cost = calculate_total_cost(instance, current_bin_types)
    
    # Índice da vizinhança atual
    k = 0
    
    # Enquanto não exploramos todas as vizinhanças
    while k < len(neighborhoods):
        # Aplica a busca local na vizinhança atual
        new_bins, new_bin_types, new_cost, _ = neighborhoods[k](instance, deepcopy(current_bins), current_bin_types.copy())
        
        # Se encontramos uma solução melhor, atualizamos a solução atual e voltamos para a primeira vizinhança
        if new_cost < current_cost:
            current_bins = new_bins
            current_bin_types = new_bin_types
            current_cost = new_cost
            k = 0
        else:
            # Se não encontramos uma solução melhor, passamos para a próxima vizinhança
            k += 1
    
    end_time = time.time()
    execution_time = end_time - start_time
    
    return current_bins, current_bin_types, current_cost, execution_time

In [32]:
# Função para verificar a validade de uma solução
def verify_solution(instance, bins, bin_types):
    """
    Verifica se uma solução é válida.
    """
    # Verifica se todos os itens estão alocados
    all_items = [item for bin_items in bins for item in bin_items]
    if len(all_items) != instance.num_items:
        print(f"Erro: Número de itens alocados ({len(all_items)}) é diferente do número de itens na instância ({instance.num_items})")
        return False
    
    # Verifica se cada item aparece exatamente uma vez
    item_count = {}
    for item in all_items:
        item_count[item] = item_count.get(item, 0) + 1
    for item, count in item_count.items():
        if count > 1:
            print(f"Erro: Item {item} aparece {count} vezes na solução")
            return False
    
    # Verifica se todos os itens estão presentes
    for i in range(instance.num_items):
        if i not in all_items:
            print(f"Erro: Item {i} não aparece na solução")
            return False
    
    # Verifica se nenhum bin excede sua capacidade
    for i, bin_items in enumerate(bins):
        bin_type = bin_types[i]
        total_weight = sum(instance.item_weights[item] for item in bin_items)
        if total_weight > instance.bin_capacities[bin_type]:
            print(f"Erro: Bin {i} excede a capacidade. Peso total: {total_weight}, Capacidade: {instance.bin_capacities[bin_type]}")
            return False
    
    return True

In [33]:
# Função para contar quantos bins de cada tipo são usados
def count_bin_types(bin_types, num_bin_types):
    """
    Conta quantos bins de cada tipo são usados na solução.
    """
    count = {i: 0 for i in range(num_bin_types)}
    for bin_type in bin_types:
        count[bin_type] += 1
    return count

In [34]:
# Função para exibir resultados
def print_results(filename, instance, bins, bin_types, total_cost, execution_time):
    """
    Exibe os resultados.
    """
    print(f"--------------------- INSTANCIA: {filename} ---------------------")
    print(f"Número de itens (n): {instance.num_items}")
    print(f"Número de tipos de bin (m): {instance.num_bin_types}")
    print()
    print("Tipos de bin (capacidade / custo):")
    for i in range(instance.num_bin_types):
        print(f"  Tipo {i+1}: {instance.bin_capacities[i]} / {instance.bin_costs[i]}")
    print()
    print("Solução encontrada:")
    print(f"  Número de bins: {len(bins)}")
    print(f"  Custo total: {total_cost}")
    
    # Conta quantos bins de cada tipo são usados
    bin_type_count = count_bin_types(bin_types, instance.num_bin_types)
    print("  Distribuição de tipos de bins:")
    for bin_type, count in bin_type_count.items():
        if count > 0:
            print(f"    Tipo {bin_type+1}: {count} bins")
    
    print(f"  Tempo de execução: {execution_time:.4f} segundos")
    print("--------------------------------")

In [35]:
# Função principal para resolver uma instância
def solve_instance(instance, filename, time_limit=60):
    """
    Resolve uma instância do VSBPP usando diferentes heurísticas e meta-heurísticas.
    """
    overall_start_time = time.time()
    
    # Gera soluções iniciais usando diferentes heurísticas
    print("Gerando solução inicial usando BFD...")
    bfd_bins, bfd_bin_types, bfd_cost, bfd_time = best_fit_decreasing(instance)
    
    print("Gerando solução inicial usando SSP...")
    ssp_bins, ssp_bin_types, ssp_cost, ssp_time = subset_sum_heuristic(instance)
    
    # Seleciona a melhor solução inicial
    if bfd_cost <= ssp_cost:
        print(f"Usando solução BFD como inicial: {len(bfd_bins)} bins, custo {bfd_cost}")
        initial_bins = bfd_bins
        initial_bin_types = bfd_bin_types
        initial_cost = bfd_cost
        initial_time = bfd_time
    else:
        print(f"Usando solução SSP como inicial: {len(ssp_bins)} bins, custo {ssp_cost}")
        initial_bins = ssp_bins
        initial_bin_types = ssp_bin_types
        initial_cost = ssp_cost
        initial_time = ssp_time
    
    print(f"Distribuição de tipos de bins: {count_bin_types(initial_bin_types, instance.num_bin_types)}")
    
    # Verifica a validade da solução inicial
    if not verify_solution(instance, initial_bins, initial_bin_types):
        print("A solução inicial é inválida!")
        return None
    
    # Aplica a meta-heurística VND se ainda temos tempo
    if time.time() - overall_start_time < time_limit:
        print("Aplicando Variable Neighborhood Descent...")
        vnd_bins, vnd_bin_types, vnd_cost, vnd_time = variable_neighborhood_descent(instance, initial_bins, initial_bin_types)
        
        print(f"Solução após VND: {len(vnd_bins)} bins, custo total {vnd_cost}")
        print(f"Distribuição de tipos de bins: {count_bin_types(vnd_bin_types, instance.num_bin_types)}")
        print(f"Melhoria: {initial_cost - vnd_cost} ({(initial_cost - vnd_cost) / initial_cost * 100:.2f}%)")
        
        # Verifica a validade da solução final
        if not verify_solution(instance, vnd_bins, vnd_bin_types):
            print("A solução final é inválida!")
            print_results(filename, instance, initial_bins, initial_bin_types, initial_cost, initial_time)
            return initial_bins, initial_bin_types, initial_cost, initial_time
        
        if vnd_cost < initial_cost:
            print_results(filename, instance, vnd_bins, vnd_bin_types, vnd_cost, initial_time + vnd_time)
            return vnd_bins, vnd_bin_types, vnd_cost, initial_time + vnd_time
    
    print_results(filename, instance, initial_bins, initial_bin_types, initial_cost, initial_time)
    return initial_bins, initial_bin_types, initial_cost, initial_time


In [36]:
# Função para exportar os resultados para um arquivo CSV
def export_results_to_csv(results, filename="../results/resultados_vsbpp.csv"):
    """
    Exporta os resultados para um arquivo CSV.
    
    Args:
        results: Lista de tuplas (filename, num_bins, cost, execution_time)
        filename: Nome do arquivo CSV
    """
    # Cria um DataFrame para os resultados
    df = pd.DataFrame(results, columns=["Instância", "Número de Bins", "Custo Total", "Tempo (s)"])
    
    # Adiciona a data e hora da execução
    now = datetime.now()
    df["Data da Execução"] = now.strftime("%Y-%m-%d %H:%M:%S")
    
    # Exporta para CSV
    df.to_csv(filename, index=False)
    print(f"Resultados exportados para {filename}")

In [ ]:
# Função principal
def main():
    # Carrega todas as instâncias
    all_instances = load_all_instances("../data/")
    
    # Inicializa uma lista para armazenar os resultados
    results = []
    
    # Se encontramos alguma instância, vamos resolver cada uma
    for filename, instance in all_instances:
        print(f"\nResolvendo instância: {filename}")
        
        # Define um limite de tempo para cada instância (30 segundos)
        solution = solve_instance(instance, filename, time_limit=30)
        
        if solution:
            bins, bin_types, total_cost, execution_time = solution
            results.append((filename, len(bins), total_cost, execution_time))
    
    # Imprime um resumo dos resultados
    print("\nResumo dos resultados:")
    print("Instância\tNúmero de Bins\tCusto Total\tTempo de Execução (s)")
    for filename, num_bins, cost, execution_time in results:
        print(f"{filename}\t{num_bins}\t{cost}\t{execution_time:.4f}")
    
    # Exporta os resultados para um arquivo CSV
    if results:
        export_results_to_csv(results)

if __name__ == "__main__":
    main()

Instância carregada: H&Sconc100-2-1.txt
Instância carregada: H&Sconc100-2-10.txt
Instância carregada: H&Sconc100-2-2.txt
Instância carregada: H&Sconc100-2-3.txt
Instância carregada: H&Sconc100-2-4.txt
Instância carregada: H&Sconc100-2-5.txt
Instância carregada: H&Sconc100-2-6.txt
Instância carregada: H&Sconc100-2-7.txt
Instância carregada: H&Sconc100-2-8.txt
Instância carregada: H&Sconc100-2-9.txt
Instância carregada: H&Sconc1000-2-1.txt
Instância carregada: H&Sconc1000-2-10.txt
Instância carregada: H&Sconc1000-2-2.txt
Instância carregada: H&Sconc1000-2-3.txt
Instância carregada: H&Sconc1000-2-4.txt
Instância carregada: H&Sconc1000-2-5.txt
Instância carregada: H&Sconc1000-2-6.txt
Instância carregada: H&Sconc1000-2-7.txt
Instância carregada: H&Sconc1000-2-8.txt
Instância carregada: H&Sconc1000-2-9.txt
Instância carregada: H&Sconc200-2-1.txt
Instância carregada: H&Sconc200-2-10.txt
Instância carregada: H&Sconc200-2-2.txt
Instância carregada: H&Sconc200-2-3.txt
Instância carregada: H&Scon